# BÀI TẬP: TITANIC
**Nguồn:** kaggle.com/c/titanic (891 dòng)


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [9]:
df.shape

(891, 16)

In [2]:
# TODO
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    str    
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    str    
 8   class        891 non-null    str    
 9   who          891 non-null    str    
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    str    
 12  embark_town  889 non-null    str    
 13  alive        891 non-null    str    
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), str(7)
memory usage: 92.4 KB


## A.2. Missing values & Duplicate data

In [3]:
# TODO
df.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [3]:
df.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
886     True
887    False
888    False
889    False
890    False
Length: 891, dtype: bool

## A.3. Invalid values

In [2]:
# TODO
invalid_age = df[df['age'] < 0]
invalid_fare = df[df['fare'] < 0]

print(invalid_age)
print(invalid_fare)


Empty DataFrame
Columns: [survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone]
Index: []
Empty DataFrame
Columns: [survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone]
Index: []


## A.4. Create a new column
Tạo cột `family_size` = sibsp + parch + 1.

In [7]:
# TODO
df['family_size'] = df['sibsp'] + df['parch'] + 1

df[['sibsp', 'parch', 'family_size']].head()

,sibsp,parch,family_size
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [12]:
# TODO
num_cols = ['age', 'fare', 'sibsp', 'parch', 'family_size']

central_tendency = pd.DataFrame({
        'Mean': df[num_cols].mean(),
        'Median': df[num_cols].median(),
        'Mode': df[num_cols].mode().iloc[0]
})

print(central_tendency.round(2))

              Mean  Median   Mode
age          29.70   28.00  24.00
fare         32.20   14.45   8.05
sibsp         0.52    0.00   0.00
parch         0.38    0.00   0.00
family_size   1.90    1.00   1.00


## Group 2 — Dispersion

In [14]:
# TODO
num_cols = ['age', 'fare', 'sibsp', 'parch', 'family_size']

sub = df[num_cols]
q1 = sub.quantile(0.25)
q3 = sub.quantile(0.75)

dispersion_df = pd.DataFrame({
        'Range': sub.max() - sub.min(),
        'Std': sub.std(),
        'IQR': q3 - q1,
        'CV': sub.std() / sub.mean()
})

print(dispersion_df.round(2))

              Range    Std    IQR    CV
age           79.58  14.53  17.88  0.49
fare         512.33  49.69  23.09  1.54
sibsp          8.00   1.10   1.00  2.11
parch          6.00   0.81   0.00  2.11
family_size   10.00   1.61   1.00  0.85


## Group 3 — Location and Shape

In [17]:
# TODO
shape_df = pd.DataFrame({
    'Skewness': df[num_cols].skew(),
    'Kurtosis': df[num_cols].kurtosis(),
    'Q1 (25%)': df[num_cols].quantile(0.25),
    'Q3 (75%)': df[num_cols].quantile(0.75),
    'Median': df[num_cols].median(),
    'P90 (90%)': df[num_cols].quantile(0.9)
})
print(shape_df.round(2))

             Skewness  Kurtosis  Q1 (25%)  Q3 (75%)  Median  P90 (90%)
age              0.39      0.18     20.12      38.0   28.00      50.00
fare             4.79     33.40      7.91      31.0   14.45      77.96
sibsp            3.70     17.88      0.00       1.0    0.00       1.00
parch            2.75      9.78      0.00       0.0    0.00       2.00
family_size      2.73      9.16      1.00       2.0    1.00       4.00


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Hạng vé nào có tỷ lệ sống sót cao nhất, chênh lệch bao nhiêu so với hạng thấp nhất?

In [19]:
# TODO
pclass_survival = df.groupby('pclass')['survived'].agg(['count','mean']).rename(columns={'mean': 'survival_rate'})
pclass_survival['survival_rate (%)'] = pclass_survival['survival_rate'] * 100

diff = pclass_survival['survival_rate'].max() - pclass_survival['survival_rate'].min()

print(pclass_survival.round(2))

        count  survival_rate  survival_rate (%)
pclass                                         
1         216           0.63              62.96
2         184           0.47              47.28
3         491           0.24              24.24


## Câu hỏi 2: Giới tính hay hạng vé ảnh hưởng đến sống sót mạnh hơn?

In [25]:
# TODO
sex_survival = df.groupby('sex')['survived'].agg(counts='count', rate = 'mean')

print(sex_survival.round(2))

        counts  rate
sex                 
female     314  0.74
male       577  0.19


In [26]:
pclass_survival = df.groupby('pclass')['survived'].agg(counts = 'count', rate = 'mean')

print(pclass_survival.round(2))

        counts  rate
pclass              
1          216  0.63
2          184  0.47
3          491  0.24


In [27]:
pivot_table = df.pivot_table(index='sex', columns='pclass', values='survived', aggfunc='mean')

print(pivot_table.round(2))

pclass     1     2     3
sex                     
female  0.97  0.92  0.50
male    0.37  0.16  0.14


## Câu hỏi 3: Vé đắt hơn có thực sự sống sót cao hơn không?

In [29]:
# TODO
fare_survival = df.groupby('survived')['fare'].agg(['count','mean','median','std'])

print(fare_survival.round(2))

          count   mean  median    std
survived                             
0           549  22.12    10.5  31.39
1           342  48.40    26.0  66.60


In [33]:
df['fare_group'] = pd.qcut(df['fare'], q=4, labels = ['Thap(Q1)', 'Trung binh thap(Q2)', 'Trung binh cao(Q3)', 'Cao(Q4)'])
fare_group_survival = df.groupby('fare_group', observed=False)['survived'].agg(rate = 'mean', counts = 'count')

print(fare_group_survival.round(2))

                     rate  counts
fare_group                       
Thap(Q1)             0.20     223
Trung binh thap(Q2)  0.30     224
Trung binh cao(Q3)   0.45     222
Cao(Q4)              0.58     222


## Câu hỏi 4: Gia đình đông người có ảnh hưởng đến khả năng sống sót không?

In [38]:
# TODO
family_survival = df.groupby('family_size')['survived'].agg(rate='mean', counts='count').sort_values(by='rate', ascending=False)

print(family_survival.round(2))

             rate  counts
family_size              
4            0.72      29
3            0.58     102
2            0.55     161
7            0.33      12
1            0.30     537
5            0.20      15
6            0.14      22
8            0.00       6
11           0.00       7


## Câu hỏi 5: Cảng lên tàu (embark_town) nào có tỷ lệ sống sót cao nhất?

In [41]:
# TODO
embark_stats = df.groupby('embark_town')['survived'].agg(total_passengers='count', rate='mean').sort_values(by='rate', ascending=False)
embark_stats['rate (%)'] = embark_stats['rate'] * 100

best_port = embark_stats.index[0]

print(best_port)

Cherbourg


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể về dữ liệu Titanic.

*(Viết insight của bạn vào đây...)*